# Late deliveries vs. customer satisfaction: charts

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def save(fig, name):
    """Show the figure and export it to img/<name>.png (needs kaleido)."""
    fig.show()
    try:
        fig.write_image(str(f"img/{name}.png"), scale=2)
    except Exception as e:  # kaleido missing or no browser available
        print(f"PNG export skipped for {name}: {' '.join(str(e).split())[:140]}")

## Setup: connection and base dataset

In [2]:
# Load database
con = duckdb.connect("olist.duckdb")
for f in ["sql/00_load.sql", "sql/02_views.sql"]:
    con.execute(Path(f).read_text())
print("DB pronto")

# Helper function to load tables
q = lambda f: con.sql(Path(f"sql/{f}").read_text()).df()

DB pronto


## Summary statistics

In [3]:
summary = q("q00_summary.sql")
summary

,n_orders,n_with_review,pct_with_review,pct_late,avg_score
0,96203,95560,99.3,6.79,4.157


## 1. How widespread is lateness?

Monthly late rate (top) and order volume (bottom).

In [4]:
monthly = q("q01_monthly_lateness.sql")
monthly["month"] = pd.to_datetime(monthly["month"])
peak = monthly.loc[monthly["perc_late_orders"].idxmax()]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.06)
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["perc_late_orders"], mode="lines+markers",
                         name="% late"), row=1, col=1)
fig.add_trace(go.Bar(x=monthly["month"], y=monthly["n_orders"], name="Orders"), row=2, col=1)
fig.update_yaxes(title_text="% late orders", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="Orders", row=2, col=1)
fig.update_layout(title="Percentage of late orders and total number of orders per month", showlegend=False)
save(fig, "late_rate_monthly")

## 2. Does delay lower the score?

Average score per delay bucket and the full score distribution confirm that delay lowers the score.

In [5]:
by_bucket = q("q02_score_by_delay.sql")

fig = px.bar(by_bucket, x="bucket", y="avg_score", color="bucket",
             text=by_bucket["n"].map("n={:,}".format))
fig.update_yaxes(range=[1, 5], title="Average review score")
fig.update_xaxes(title=None)
fig.update_layout(title="Average score by delay bucket",
                  showlegend=False)
save(fig, "score_by_delay")

In [6]:
dist = q("q03_delay_detail.sql")

fig = px.bar(dist, x="bucket", y="pct", color="score", barmode="stack")
fig.update_yaxes(title="% of reviews")
fig.update_xaxes(title=None)
fig.update_layout(title="Distributions of score for each delay bucket")
save(fig, "score_distribution_by_delay")

## 3. Where? States

Customer state: late rate against average score, bubble size = orders. Dotted lines are the overall values.

In [7]:
states = q("q04_delivery_state.sql")

fig = px.scatter(states, x="perc_late_orders", y="avg_score", size="n_orders", text="state", hover_name="state",
                 color="state", size_max=40)
fig.update_traces(textposition="top center")
fig.update_xaxes(title="% late orders")
fig.update_yaxes(title="Average review score")
fig.update_layout(title="States: late rate vs average score", showlegend=False)
save(fig, "state_scatter")

## 4. On whom to intervene? Sellers

Grain is (order, seller): an order with items from several sellers counts for each of them. The curve shows how concentrated late orders are; the table ranks sellers by late rate, among those with at least 30 orders.

In [8]:
sellers = q("q05_sellers_ranking.sql")

ranked = sellers.sort_values("late_orders", ascending=False).reset_index(drop=True)
ranked["seller_share"] = 100 * (ranked.index + 1) / len(ranked)
ranked["cum_late_share"] = 100 * ranked["late_orders"].cumsum() / ranked["late_orders"].sum()
k50 = int((ranked["cum_late_share"] >= 50).idxmax()) + 1

fig = go.Figure()
fig.add_trace(go.Scatter(x=ranked["seller_share"], y=ranked["cum_late_share"], mode="lines",
                         name="Cumulative share of late orders"))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color="grey", dash="dot"),
                         name="Even distribution"))
fig.update_xaxes(title="% of sellers (ranked by number of late orders)")
fig.update_yaxes(title="% of late orders (cumulative)")
fig.update_layout(title=f"{k50} sellers ({100 * k50 / len(ranked):.1f}% of {len(ranked)}) account for half of late orders")
save(fig, "seller_pareto")

top_rate = (sellers
            .assign(pct_late=lambda d: (100 * d["late_orders"] / d["n_orders"]).round(1))
            .sort_values("pct_late", ascending=False)
            .head(10))
top_rate

,seller_id,customer_state,perc_late_orders,late_orders,n_orders,rank_in_state,pct_late
22,88460e8ebdecbfecb5f9601833981930,MG,36.111111,13.0,31,1,41.9
443,8160255418d5aaa7dbdc9f4c64ebda44,RJ,30.555556,22.0,59,1,37.3
444,88460e8ebdecbfecb5f9601833981930,RJ,28.888889,13.0,36,2,36.1
445,897060da8b9a21f655304d50fd935913,RJ,25.000000,14.0,39,3,35.9
446,391fc6631aebcf3004804e51b40bcf1e,RJ,25.000000,17.0,61,3,27.9
451,1835b56ce799e6a4dc4eddc053f04066,RJ,20.000000,18.0,66,8,27.3
449,1f50f920176fa81dab994f9023523100,RJ,20.567376,58.0,219,7,26.5
89,99a54764c341d5dc80b4a8fac4eba3fb,SP,15.686275,8.0,31,4,25.8
447,7ddcbb64b5bc1ef36ca8c151f6ec77df,RJ,24.000000,12.0,47,5,25.5
448,8b321bb669392f5163d04c59e235e066,RJ,22.758621,33.0,134,6,24.6


## 5. Is it a category artifact?

Per category: average score for on-time orders vs orders 8+ days late (grain: order and category, so a multi-category order counts in each). Only categories with at least 30 reviewed orders in both groups.

In [9]:
cats = q("q06_category_confounders.sql")
cats["delta"] = cats["avg_score_8_plus_days_late"] - cats["avg_score_on_time"]
cats = cats.sort_values("delta").reset_index(drop=True).head(20)   # show only 20 - biggest drop first
n_drop = int((cats["delta"] < 0).sum())

xs, ys = [], []
for _, r in cats.iterrows():
    xs += [r["avg_score_8_plus_days_late"], r["avg_score_on_time"], None]
    ys += [r["category"], r["category"], None]

fig = go.Figure()
fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#cccccc", width=2), showlegend=False))
fig.add_trace(go.Scatter(x=cats["avg_score_on_time"], y=cats["category"], mode="markers",
                         marker=dict(color="grey", size=9), name="On time"))
fig.add_trace(go.Scatter(x=cats["avg_score_8_plus_days_late"], y=cats["category"], mode="markers",
                         marker=dict(color="red", size=9), name="8+ days late"))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="Average review score")
fig.update_layout(title="Score drop with 8+ days late for every category (shows 20 biggest drop)",
                  height=max(400, 24 * len(cats) + 140))
save(fig, "category_dumbbell")

## Caveats

- Association, not causation: late orders also differ in distance, product type and seller.
- Only customers who left a review are in the satisfaction metrics.
- Delivery metrics cover delivered orders only.
- Small groups are excluded (orders < 30)

In [10]:
con.close()